# Amazon-Reviewer Persona Attribute Extraction

Extract persona **attributes** (values + supporting evidence/descriptions) for a
single Amazon shopper from their **entire product-review history**, using
Qwen3.6-35B-A3B served with **vLLM** — the same model/schema as the wiki
extractor, but a **new prompt** designed for review histories instead of
biographies.

Pipeline:
1. Configure caches + HF access (all on `/n/netscratch/lu_lab/Lab/xiaominli`).
2. Load the **selection index** (`data/amazon/selected_users_100k.parquet`) from
   `explore_amazon_data.ipynb`.
3. Assemble each user's reviews into a `profile_text` (the canonical format).
4. Load the persona dimension schema (`persona/schema/dimensions.json`).
5. Build the **Amazon** extraction prompt (`build_amazon_prompt`).
6. Load Qwen3.6-35B-A3B with vLLM.
7. Run an extraction demo on a few shoppers and parse the JSON output.

> Kernel: **env05** (`/n/home08/xiaominli/.conda/envs/env05`). vLLM >= 0.24.0 is
> required for the `qwen3_5_moe` architecture. The demo (steps 6-7) needs an H200;
> steps 1-5 are CPU-only.


## 0. Configuration

Everything cache-heavy is pinned under `/n/netscratch/lu_lab/Lab/xiaominli` so
nothing lands on the home filesystem.


In [ ]:
import os
import re
import json
from pathlib import Path

# --- Cache locations (must live on the netscratch filesystem) ---
NETSCRATCH = Path("/n/netscratch/lu_lab/Lab/xiaominli")
CACHE_ROOT = NETSCRATCH / "mycache"
HF_HOME = CACHE_ROOT / "hf_home"
HF_HUB_CACHE = HF_HOME / "hub"
HF_XET_CACHE = HF_HOME / "xet"
VLLM_DOWNLOAD_DIR = HF_HUB_CACHE
for p in (HF_HOME, HF_HUB_CACHE, HF_XET_CACHE):
    p.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["HF_XET_CACHE"] = str(HF_XET_CACHE)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

# --- HF token (gated dataset). Prefer env; else read HF_TOKEN_matraix from ~/.bashrc (never printed). ---
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF_TOKEN_matraix")
if not HF_TOKEN:
    for line in Path(os.path.expanduser("~/.bashrc")).read_text().splitlines():
        m = re.search(r"HF_TOKEN_matraix=[\'\"]?([^\'\"\s]+)", line)
        if m:
            HF_TOKEN = m.group(1)
            break
print("HF token loaded:", bool(HF_TOKEN))

# --- Repo / dataset / schema locations ---
DATASET_REPO = "MatrAIx2026/MatrAIx2026"
BASE = "amazon/modal_artifacts"
UBUK = f"{BASE}/amazon_reviews_2018_2023_user_buckets_min30_verified70_text2000"
REPO_ROOT = NETSCRATCH / "LLMResearch/MatrAIx"
DIMENSIONS_JSON = REPO_ROOT / "persona/schema/dimensions.json"
DATA_DIR = REPO_ROOT / "persona/human_extraction/data"
SELECTION = DATA_DIR / "amazon/selected_users_100k.parquet"
MODEL_ID = "Qwen/Qwen3.6-35B-A3B"

from huggingface_hub import HfApi, hf_hub_download  # noqa: E402
api = HfApi(token=HF_TOKEN)
def dl(fn):
    return hf_hub_download(DATASET_REPO, fn, repo_type="dataset", token=HF_TOKEN)
print("selection exists:", SELECTION.exists())


## 1. Load the selection index

The ~100K shoppers chosen in `explore_amazon_data.ipynb` (length-filtered to fit
the Qwen 32k context and drop too-short histories). Each row is one user +
pre-computed length stats; the raw reviews live in `user_buckets`.


In [ ]:
import pandas as pd
import numpy as np

sel = pd.read_parquet(SELECTION)
print(f"selected users : {len(sel):,}")
print("columns        :", list(sel.columns))
print("buckets        :", sel.user_bucket.nunique(), "(00..ff)")
print("est_tokens     : p50=%d  p90=%d  max=%d" % (
    np.percentile(sel.est_tokens, 50), np.percentile(sel.est_tokens, 90), sel.est_tokens.max()))
sel.head(3)


## 2. Assemble a user's `profile_text`

One **persona = one user**. We pull all of a user's reviews from their
`user_bucket` (across every product-category file), sort chronologically, and
format them into a single `profile_text`. This is the exact text the extractor
sees; the production script must reuse `assemble_profile` verbatim so token
budgets and prompts match this notebook.


In [ ]:
REVIEW_TMPL = ("[{date}] {category} | {parent_asin} | rating={rating:.0f}/5 | "
               "verified={verified}\nTitle: {title}\n{text}")
MAX_PROFILE_CHARS = 48000   # safety cap; selection already keeps est_tokens<=16k

def assemble_profile(g: pd.DataFrame) -> str:
    """Concatenate one user's reviews (chronological) into a profile_text."""
    g = g.sort_values("timestamp")
    parts = [REVIEW_TMPL.format(
                date=r.date, category=r.category, parent_asin=r.parent_asin,
                rating=float(r.rating), verified=bool(r.verified_purchase),
                title=(r.title or ""), text=(r.text or ""))
             for r in g.itertuples()]
    header = (f"Amazon reviewer profile — {len(g)} reviews across "
              f"{g.category.nunique()} categories.\n\n")
    return (header + "\n\n".join(parts))[:MAX_PROFILE_CHARS]

def load_bucket_reviews(b: str) -> pd.DataFrame:
    """All reviews in one user_bucket (across every category file)."""
    files = [f for f in api.list_repo_files(DATASET_REPO, repo_type="dataset")
             if f.startswith(f"{UBUK}/bucket={b}/") and f.endswith(".parquet")]
    return pd.concat([pd.read_parquet(dl(f)) for f in files], ignore_index=True)


In [ ]:
# Load one bucket and assemble profiles for the selected users in it
SAMPLE_BUCKET = "00"
sel_b = sel[sel.user_bucket == SAMPLE_BUCKET]
rev = load_bucket_reviews(SAMPLE_BUCKET)
rev_sel = rev[rev.user_id.isin(set(sel_b.user_id))]
profiles = {uid: assemble_profile(g) for uid, g in rev_sel.groupby("user_id", sort=False)}
print(f"bucket={SAMPLE_BUCKET}: {len(sel_b):,} selected users, "
      f"{len(rev):,} reviews, assembled {len(profiles):,} profiles")

# peek at one assembled profile
uid0 = next(iter(profiles))
print(f"\n--- profile_text for {uid0} (head) ---\n")
print(profiles[uid0][:900])


## 3. Load the persona dimension schema

Same 1,290-dimension schema as wiki. We group by `category` so the prompt can be
chunked (the whole schema is too large for one prompt); each chunk carries the
**same full profile** plus a different slice of dimensions, then the per-chunk
field lists are merged into one persona.


In [ ]:
from collections import OrderedDict

with open(DIMENSIONS_JSON) as fh:
    schema_doc = json.load(fh)
DIMENSIONS = schema_doc["dimensions"]
print("schema:", schema_doc.get("name"), "| total dimensions:", len(DIMENSIONS))

by_category = OrderedDict()
for d in DIMENSIONS:
    by_category.setdefault(d.get("category", "Uncategorized"), []).append(d)
print("categories:", len(by_category))
for cat, dims in list(by_category.items())[:8]:
    print(f"  {cat:34s} {len(dims):3d} dims  e.g. {dims[0]['id']}")

def cat_chunks(by_cat, per_chunk=50):
    out = []
    for dims in by_cat.values():
        for i in range(0, len(dims), per_chunk):
            out.append(dims[i:i+per_chunk])
    return out

CHUNKS = cat_chunks(by_category, 50)
print("\nchunks/profile:", len(CHUNKS))


## 4. The Amazon extraction prompt (new)

Designed for **review histories**, not biographies. The model infers each
attribute from three signal types — **what** they buy (categories/products),
**how** they write (tone, detail, sentiment), and **what** they state about
themselves — and must judge the history *as a whole* (one-off items may be
gifts). The output contract (`field_id, value, confidence, evidence,
description, assignment_type`) is **identical** to the wiki extractor, so the
same `parse_fields`, schema, and scorer apply unchanged.

`assignment_type` is re-grounded for Amazon:
- **direct** — the reviewer explicitly states it about themselves.
- **structured_claim** — strongly implied by concrete purchase facts (e.g.
  repeatedly buying baby products → has a young child).
- **summary_inference** — softer inference from overall pattern / tone / style.
- **unsupported** — not supported by the reviews.


In [ ]:
def build_amazon_prompt(profile_text: str, dimensions: list) -> str:
    """Amazon-reviewer persona-extraction prompt.

    Input is ONE shopper's full review history (chronological). Persona
    attributes are inferred from what they buy, how they write, and any facts
    they state about themselves. Output schema matches the wiki extractor.
    """
    lines = [
        "You are building a persona for a single Amazon shopper from their "
        "complete product-review history.",
        "",
        "The input is a chronological list of that ONE person's reviews. Each "
        "review has a date, product category, product id (ASIN), star rating, a "
        "verified-purchase flag, a title, and body text. Infer who this shopper "
        "is from the WHOLE history together:",
        "- WHAT they buy: product categories and specific items reveal interests, "
        "hobbies, life stage, household, budget, and needs.",
        "- HOW they write: tone, length, detail, sentiment, and vocabulary reveal "
        "personality, values, and writing style.",
        "- WHAT they say: facts a reviewer states about themselves (\"as a "
        "nurse\", \"for my kids\", \"at 65 I...\") are the strongest signal.",
        "",
        "Return ONLY JSON with this shape (no markdown, no commentary):",
        '{"fields": [{"field_id": "<one id from DIMENSIONS below>", '
        '"value": "<one allowed value, copied verbatim, or null>", '
        '"confidence": 0.0, '
        '"evidence": "<short quote copied verbatim from one review>", '
        '"description": "<1-2 sentence description of this shopper for this attribute>", '
        '"assignment_type": "direct"}]}',
        "",
        "assignment_type values (Amazon context):",
        "- direct: the reviewer explicitly states it about themselves in a review.",
        "- structured_claim: strongly implied by concrete purchase facts (e.g. "
        "repeatedly buying baby products -> has a young child).",
        "- summary_inference: a softer inference from the overall pattern, tone, "
        "or writing style across many reviews.",
        "- unsupported: not supported by the reviews.",
        "",
        "Rules:",
        "- Emit exactly one object per dimension listed below.",
        "- value MUST be exactly one of that dimension's allowed values (copied "
        "verbatim), OR null.",
        "- Judge the history as a whole; prefer attributes backed by MULTIPLE "
        "reviews over a single purchase (one-off items may be gifts for others).",
        "- If the reviews do not support a dimension, set value to null, "
        'assignment_type to "unsupported", and description to "".',
        "- Every non-null value MUST include a short evidence quote copied "
        "verbatim from one of the reviews.",
        "- description: 1-2 concrete sentences describing THIS shopper for this "
        "attribute using details from their reviews (categories, products, "
        "statements). Describe the person; do not justify the label.",
        "- Be conservative with sensitive attributes (age, gender, health, "
        "ethnicity, religion, income): assign only when clearly stated or very "
        "strongly implied; otherwise null/unsupported.",
        "- Return valid JSON only, with no markdown.",
        "",
        "DIMENSIONS (field_id — label — description — allowed values):",
    ]
    for d in dimensions:
        allowed = " | ".join(str(v) for v in d.get("values", [])) or "(free value)"
        desc = str(d.get("description", "")).strip()
        lines.append(f"- {d['id']} — {d.get('label', d['id'])} — {desc} — [{allowed}]")
    lines += ["", "REVIEWER HISTORY:", profile_text]
    return "\n".join(lines)


def parse_fields(text: str) -> list:
    """Same tolerant parser as the production script (handles fields:null)."""
    s, e = text.find("{"), text.rfind("}")
    if s == -1 or e == -1:
        return []
    try:
        obj = json.loads(text[s:e+1])
    except json.JSONDecodeError:
        return []
    if not isinstance(obj, dict):
        return []
    f = obj.get("fields")
    return f if isinstance(f, list) else []

# preview the prompt for one profile + one small category chunk
demo_chunk = by_category[list(by_category)[0]][:6]
print(build_amazon_prompt(profiles[uid0], demo_chunk)[:1800])


## 5. Load Qwen3.6-35B-A3B with vLLM

H200 (143 GB) fits the 35B MoE (~70 GB bf16). First run downloads + loads the
model (a few minutes). Requires vLLM >= 0.24.0 for `qwen3_5_moe`. **GPU cell.**


In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=str(HF_HUB_CACHE))
llm = LLM(
    model=MODEL_ID,
    download_dir=str(VLLM_DOWNLOAD_DIR),
    dtype="bfloat16",
    gpu_memory_utilization=0.90,
    max_model_len=32768,
    enable_prefix_caching=True,   # profile+dims prefix is shared across chunks
    trust_remote_code=True,
)
sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=8192)
print("vLLM ready.")


## 6. Extraction demo

Run a few shoppers × one category chunk through vLLM, then parse the returned
attribute/value/description/evidence fields. Swap `CATEGORY` for any key in
`by_category`, or loop over `CHUNKS` to extract the full schema per user.

> Qwen3 is a hybrid reasoning model; we disable thinking (`enable_thinking=False`)
> for clean JSON. Drop it if you want chain-of-thought.


In [ ]:
N_USERS = 3
CATEGORY = "Demographic: Core"   # any key in by_category
dims = by_category.get(CATEGORY, next(iter(by_category.values())))
demo_uids = list(profiles)[:N_USERS]

prompts = []
for uid in demo_uids:
    msg = build_amazon_prompt(profiles[uid], dims)
    prompts.append(tokenizer.apply_chat_template(
        [{"role": "user", "content": msg}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False))

outputs = llm.generate(prompts, sampling_params)
print("Generated", len(outputs), "responses for category:", CATEGORY)


In [ ]:
# Parse and display the extracted attributes
records = []
for uid, out in zip(demo_uids, outputs):
    for f in parse_fields(out.outputs[0].text):
        records.append({
            "user_id": uid,
            "field_id": f.get("field_id"),
            "value": f.get("value"),
            "assignment_type": f.get("assignment_type"),
            "confidence": f.get("confidence"),
            "description": f.get("description"),
            "evidence": f.get("evidence"),
        })
res = pd.DataFrame(records)
pd.set_option("display.max_colwidth", 90)
print(f"{len(res)} fields from {len(demo_uids)} shoppers")
res[res.value.notna()].head(30)


## 7. Notes / next steps

- **Unit = user.** `profile_text` = all of a shopper's reviews (chronological)
  via `assemble_profile`; reuse it verbatim in the production script so budgets
  match. Selection already caps `est_tokens<=16k` so the 48k-char safety cap
  almost never triggers.
- **New prompt.** `build_amazon_prompt` reframes extraction around purchase
  behaviour + writing style while keeping the wiki output contract (same
  `parse_fields`, schema, scorer). `assignment_type` is re-grounded for Amazon.
- **Full run.** To productionise, mirror `scripts/run_extraction.py` +
  `jobs/extract_shard.job`: shard the 100K selection by `user_bucket`, load each
  bucket's reviews once, assemble per user, run all `CHUNKS`, merge fields, and
  append to `data/amazon/extraction_v1/shard_XXXX.jsonl` (resumable by user_id).
- **Cost.** 100K users × ~53 chunks is far smaller than the 2.13M wiki run;
  bucket-local review loading (one bucket = ~6.9k users) keeps I/O cheap.
